# Israeli legal eval: 500 questions + 10 paralegal cases, on Kaggle

Runs `legal_txt/Evals/israeli_legal_eval` -- 500 Hebrew questions (`questions.jsonl`) across 9
LegalBench-style categories and 11 areas of law, plus 10 fictional paralegal cases (`cases/`) --
against the **bulk legal corpus** (`data/legal_corpus_vectordb`: Israeli laws + regulations,
`scripts/legal_data/`), not the Legal tab's small signed-bundle index (too narrow for a set this
broad). See `legal_txt/Evals/israeli_legal_eval/README.md` for what the set covers and its caveats.

RAG only: every question and case is answered with retrieval over the bulk corpus first
(`scripts/legal_data/eval_run.py::retrieve`), same as the app's own retrieval path. (The scripts
still support `--mode no_context`, but this notebook no longer runs or reports it.)

**Detailed reasoning, for fine-tuning**: every model call -- answering and judging -- runs with
thinking enabled and is logged in full (prompt, reasoning, output, timing) to
`<logging.llm_trace_dir>/*.jsonl` (`llm/trace.py`), one file per day, each line tagged with the
question/case id (`job_id`). `scripts/llm_trace_report.py` renders it to Markdown, one section per
call. Both the raw JSONL and the rendered Markdown go to Output.

**Score report**: the 500 questions are scored by `legal_txt/Evals/israeli_legal_eval/score.py`
(unmodified -- provided with the eval set): exact-match for the 80 multiple-choice items, label
match for the 90 yes/no items, and an LLM judge (`eval_run.py judge`, a stronger/different model
than the one under test) against the gold answer for the rest, with a hallucination rate and a
per-category/per-area/per-confidence breakdown. The 10 cases are scored against their 100-point
rubric (`cases_gold.json`) by `eval_cases.py`, judge model again, Prompt B from `judge_prompt.md`.

## Before running (panel on the right)

1. **Build the corpus first, if you haven't**: `notebooks/kaggle_legal_corpus_build.ipynb` (a
   separate run) produces `legal_corpus_vectordb.zip`. Upload it as a **private Kaggle Dataset**
   and attach it here (*Add Input*) -- this notebook does not rebuild the corpus.
2. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **On**.
3. *Add-ons -> Secrets*: tick **GITHUB_TOKEN** (a token that can read `zananiri/AI-IZ`).
4. Set `BRANCH`, the models, and `LIMIT`/`RUN_CASES` in the next cell -- **read the time
   budget note there first**: the full 500 questions may not fit in one Kaggle session.
5. *Save Version -> Save & Run All (Commit)*. Runs in the background; download
   `legal_eval_bulk500_results.zip` from the version's *Output* tab when it finishes.

## Resuming across sessions

Every step here skips ids it already has an answer/judgement for (`--out` files are read back in
first). To continue a run that hit Kaggle's session limit: *Add Input* that earlier version's own
Output, set `RESUME_FROM` to where it lands under `/kaggle/input/...`, and run again with the same
(or a larger) `LIMIT` -- already-done ids are skipped, not re-billed.

In [ ]:
BRANCH = "main"            # the branch to test
ANSWER_MODEL = "qwen3:8b"  # the model under test
JUDGE_MODEL = "qwen3:14b"  # the grader -- stronger/different from the model under test, per judge_prompt.md
MODES = ["rag"]                   # RAG only -- no no_context (model-alone) runs
RUN_CASES = True                  # the 10 paralegal cases (slower per item, but only 10)

# Time budget: on a single T4, one answer call (RAG, thinking on) has run 30-90s here, one judge
# call 15-40s, and a case work file (longer output, longer context) several minutes. The full
# set is roughly: 500 answers + ~420 judged (mcq is scored without a judge), plus 10
# cases x (answer + judge). At the low end that is several hours -- possibly more than
# Kaggle's ~9-12h session limit in one run. LIMIT below takes a sample stratified
# across the 9 categories (all 10 cases still run in full -- they're the highest-value, lowest-
# count items). Set LIMIT = None only once you know your per-call timing from a small run (the
# per-question line this notebook prints includes its seconds).
LIMIT = 60
TOP_K = 8            # retrieved chunks per question, "rag" mode
CASE_TOP_K = 16       # retrieved chunks per case, "rag" mode (a case spans several legal issues)

RESUME_FROM = None    # an earlier session's Output (Add Input -> its /kaggle/input/... path) to continue

## 1. Code and packages

In [ ]:
import os, pathlib, shutil, subprocess, json
from kaggle_secrets import UserSecretsClient

def sh(command):
    """Runs a shell command with its output streamed live; raises (stopping the run) on failure --
    unlike `!command`, which would silently carry on."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"exit code {proc.returncode}: {command}")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/AI-IZ"
!git clone -q --depth 1 -b {BRANCH} https://{token}@github.com/zananiri/AI-IZ.git {REPO}
if not pathlib.Path(REPO, ".git").exists():
    raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret and BRANCH")
!git -C {REPO} remote set-url origin https://github.com/zananiri/AI-IZ.git
%cd {REPO}
sh("git log --oneline -1")
sh('pip install -q -e ".[legal,legal-data,dev]"')

## 2. Ollama with the answer and judge models

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "2"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
!ollama pull {ANSWER_MODEL}
!ollama pull {JUDGE_MODEL}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. The bulk legal corpus (attached input) and the eval set (from the repo)

`legal_txt/Evals/israeli_legal_eval.zip` is committed in the repo -- no separate dataset needed for
it. The corpus vectordb is too big for git, so it comes from the dataset attached in step 1.

In [ ]:
input_zips = sorted(pathlib.Path("/kaggle/input").rglob("legal_corpus_vectordb*.zip"))
input_dirs = [p for p in pathlib.Path("/kaggle/input").rglob("*")
              if p.is_dir() and any((p / c).is_dir() for c in ("laws", "procedural_rules"))]
assert input_zips or input_dirs, "attach the corpus dataset built by notebooks/kaggle_legal_corpus_build.ipynb (Add Input)"
if input_zips:
    sh(f'python scripts/legal_data/install_corpus.py "{input_zips[0]}" --replace')
else:
    # A dataset Kaggle unzipped on upload (categories found directly, not a zip): no env var
    # overrides legal.corpus.vectordb_dir (only the LLM backend settings do, config.py's
    # _LLM_ENV_OVERRIDES), so copy it to where config/config.yaml's default
    # (./data/legal_corpus_vectordb) already points, instead of the app silently not finding it.
    VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
    if not VECTORDB.exists():
        shutil.copytree(input_dirs[0], VECTORDB)
    print("using unpacked corpus at", VECTORDB)

EVAL_DIR = pathlib.Path("legal_txt/Evals")
sh(f'python -c "import zipfile; zipfile.ZipFile(\'{EVAL_DIR}/israeli_legal_eval.zip\').extractall(\'{EVAL_DIR}\')"')
EVAL = EVAL_DIR / "israeli_legal_eval"
assert (EVAL / "questions.jsonl").exists(), f"eval set not found under {EVAL}"
print(EVAL, "->", sorted(p.name for p in EVAL.iterdir()))

## 4. Point the app at Ollama, and raise LLM timeouts

Case work files and RAG-context answers run longer than the app's own turns; the config's default
`request_timeout_s` (600s) has been enough on a T4 in testing, but this gives it headroom rather
than losing a slow call this far into a multi-hour run.

In [ ]:
import yaml

os.environ.update({
    "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama", "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
    "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": ANSWER_MODEL,
    "DOCSLIDES_LEGAL_JUDGE_MODEL": JUDGE_MODEL,
    "DOCSLIDES_LLM_TRACE_DIR": "./data/legal/eval/bulk500/llm_trace",  # full prompts + reasoning, per call
})

cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))
cfg["legal"]["orchestrator"]["request_timeout_s"] = 1200
cfg["legal"]["orchestrator"]["max_model_len"] = 16384  # T4s have room; the thinking pass needs its budget
pathlib.Path("config/kaggle_eval.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
os.environ["DOCSLIDES_CONFIG"] = "config/kaggle_eval.yaml"
pathlib.Path("data/legal/eval/bulk500").mkdir(parents=True, exist_ok=True)
pathlib.Path("data/legal/eval/cases").mkdir(parents=True, exist_ok=True)

## 5. Quick check (about a minute; informational -- a failure here doesn't stop the run)

In [ ]:
!python -m pytest tests/unit -q -p no:cacheprovider 2>&1 | tail -5

## 6. Stratified sample (if `LIMIT` is set)

Takes a proportional slice of each of the 9 categories rather than the first `LIMIT` rows, so a
partial run still covers every category (`questions.jsonl` is grouped by category, so "first N"
alone would skip whichever categories sort last).

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

all_questions = load_jsonl(EVAL / "questions.jsonl")
if LIMIT and LIMIT < len(all_questions):
    from collections import defaultdict
    by_cat = defaultdict(list)
    for q in all_questions:
        by_cat[q["category"]].append(q)
    share = LIMIT / len(all_questions)
    sample = [q for qs in by_cat.values() for q in qs[: max(1, round(len(qs) * share))]]
    QUESTIONS = EVAL_DIR / "questions_sample.jsonl"
    with open(QUESTIONS, "w", encoding="utf-8") as f:
        for q in sample:
            f.write(json.dumps(q, ensure_ascii=False) + "\n")
    print(f"sampled {len(sample)} / {len(all_questions)} questions across {len(by_cat)} categories -> {QUESTIONS}")
else:
    QUESTIONS = EVAL / "questions.jsonl"
    print(f"running the full set: {len(all_questions)} questions")

## 7. Answer the questions -- one configuration at a time (resumable)

Each line logs `id (category): Ns`, so you can read off real per-item timing early and Ctrl+C /
stop the run (or just let a later cell time out) if the full set won't fit this session.

In [ ]:
ANSWERS = {}
for mode in MODES:
    out = pathlib.Path(f"data/legal/eval/bulk500/answers_{mode}.jsonl")
    if RESUME_FROM:
        prev = pathlib.Path(RESUME_FROM) / f"bulk500/answers_{mode}.jsonl"
        if prev.exists() and not out.exists():
            shutil.copy(prev, out)
    print(f"=== answering, mode={mode} ===")
    sh(f'python scripts/legal_data/eval_run.py answer --mode {mode} --questions {QUESTIONS} '
       f'--top-k {TOP_K} --out {out}')
    ANSWERS[mode] = out

## 8. Judge requests + judge (score.py `prepare` is unmodified, from the eval set)

In [ ]:
SCORE_PY = EVAL / "score.py"
GOLD = EVAL / "gold.jsonl"
JUDGED = {}
for mode in MODES:
    requests = pathlib.Path(f"data/legal/eval/bulk500/judge_requests_{mode}.jsonl")
    judged = pathlib.Path(f"data/legal/eval/bulk500/judged_{mode}.jsonl")
    if RESUME_FROM:
        prev = pathlib.Path(RESUME_FROM) / f"bulk500/judged_{mode}.jsonl"
        if prev.exists() and not judged.exists():
            shutil.copy(prev, judged)
    print(f"=== judging, mode={mode} ===")
    sh(f'python {SCORE_PY} prepare --questions {QUESTIONS} --gold {GOLD} --answers {ANSWERS[mode]} --out {requests}')
    sh(f'python scripts/legal_data/eval_run.py judge --requests {requests} --out {judged}')
    JUDGED[mode] = judged

## 9. Score report (score.py `report`, unmodified)

In [ ]:
for mode in MODES:
    print(f"\n{'=' * 20} {mode} {'=' * 20}")
    sh(f'python {SCORE_PY} report --gold {GOLD} --answers {ANSWERS[mode]} --judged {JUDGED[mode]} '
       f'--json-out data/legal/eval/bulk500/report_{mode}.json')

## 10. The 10 paralegal cases: answer, judge, report

All 10 run regardless of `LIMIT` (their own time cost, per case, is the main constraint -- see the
time budget note in cell 2). RAG only.

In [ ]:
if RUN_CASES:
    for mode in MODES:
        print(f"=== cases, mode={mode} ===")
        answers = pathlib.Path(f"data/legal/eval/cases/answers_{mode}.jsonl")
        judged = pathlib.Path(f"data/legal/eval/cases/judged_{mode}.jsonl")
        if RESUME_FROM:
            for name, dest in (("answers", answers), ("judged", judged)):
                prev = pathlib.Path(RESUME_FROM) / f"cases/{name}_{mode}.jsonl"
                if prev.exists() and not dest.exists():
                    shutil.copy(prev, dest)
        sh(f'python scripts/legal_data/eval_cases.py answer --dir {EVAL} --mode {mode} '
           f'--top-k {CASE_TOP_K} --out {answers}')
        sh(f'python scripts/legal_data/eval_cases.py judge --dir {EVAL} --answers {answers} --out {judged}')
        sh(f'python scripts/legal_data/eval_cases.py report --judged {judged} '
           f'--out data/legal/eval/cases/report_{mode}.md --json-out data/legal/eval/cases/report_{mode}.json')
else:
    print("RUN_CASES is False -- skipped")

## 11. Reasoning, rendered to Markdown

`scripts/llm_trace_report.py` (already used by `notebooks/kaggle_legal_eval.ipynb`) turns the raw
trace JSONL into one Markdown section per call: reasoning, output, timing, stop reason. This is the
file to read for fine-tuning -- where the model's reasoning drifted from a right answer.

In [ ]:
trace_files = sorted(pathlib.Path("data/legal/eval/bulk500/llm_trace").glob("*.jsonl"))
if trace_files:
    sh(f'python scripts/llm_trace_report.py {" ".join(str(p) for p in trace_files)} '
       f'> data/legal/eval/bulk500/reasoning_report.md')
    calls = sum(1 for f in trace_files for _ in open(f, encoding="utf-8"))
    print(f"{calls} calls logged across {len(trace_files)} day(s)")
else:
    print("no trace files -- did any answer/judge cell actually run a model call?")

## 12. Package for Output

In [ ]:
DEST = "/kaggle/working/legal_eval_bulk500_results.zip"
!cd /kaggle/working && zip -qr {DEST} AI-IZ/data/legal/eval AI-IZ/config/kaggle_eval.yaml ollama.log
!ls -lh {DEST}

for mode in MODES:
    path = f"data/legal/eval/bulk500/report_{mode}.json"
    if pathlib.Path(path).exists():
        r = json.load(open(path, encoding="utf-8"))
        print(f"\n{mode}: overall {100 * r['overall']:.1f}% ({len(r['rows'])} scored)")
if RUN_CASES:
    for mode in MODES:
        path = f"data/legal/eval/cases/report_{mode}.json"
        if pathlib.Path(path).exists():
            r = json.load(open(path, encoding="utf-8"))
            print(f"cases ({mode}): average {r['average_total']:.1f} / 100")